![image](https://raw.githubusercontent.com/IBM/watson-machine-learning-samples/master/cloud/notebooks/headers/watsonx-Prompt_Lab-Notebook.png)
# Use watsonx, and Mistral `mistral-small-3-1-24b-instruct-2503` to analyze eXtensive Business Reporting Language (XBRL) tags of financial reports

#### Disclaimers

- Use only Projects and Spaces that are available in watsonx context.


## Notebook content

This notebook contains the steps and code to demonstrate support of tag entity extraction in watsonx. It introduces commands for data retrieval, model testing and scoring.

Some familiarity with Python is helpful. This notebook uses Python 3.11.


## Learning goal

The goal of this notebook is to demonstrate how to use `mistral-small-3-1-24b-instruct-2503` model to analyze XBRL tags of financial phrases.


## Contents

This notebook contains the following parts:

1. [Set up the environment](#Set-up-the-environment)
2. [Data loading](#Data-loading)
3. [Foundation Models on watsonx.ai](#Foundation-Models-on-watsonx.ai)
4. [Analyze the sentiment](#Analyze-the-sentiment)
5. [Score the model](#Score-the-model)
6. [Summary and next steps](#Summary-and-next-steps)

<a id="Set-up-the-environment"></a>
## Set up the environment

Before you use the sample code in this notebook, you must perform the following setup tasks:

-  Create a <a href="https://cloud.ibm.com/catalog/services/watsonxai-runtime" target="_blank" rel="noopener no referrer">watsonx.ai Runtime Service</a> instance (a free plan is offered and information about how to create the instance can be found <a href="https://dataplatform.cloud.ibm.com/docs/content/wsj/getting-started/wml-plans.html?context=wx&audience=wdp" target="_blank" rel="noopener no referrer">here</a>).


### Install and import dependencies
**Note:** `ibm-watsonx-ai` documentation can be found <a href="https://ibm.github.io/watsonx-ai-python-sdk/index.html" target="_blank" rel="noopener no referrer">here</a>.

In [1]:
%pip install "datasets==3.0.0" | tail -n 1
%pip install "scikit-learn==1.3.2" | tail -n 1
%pip install -U ibm-watsonx-ai | tail -n 1

### Defining the watsonx.ai credentials
This cell defines the watsonx.ai credentials required to work with watsonx Foundation Model inferencing.

**Action:** Provide the IBM Cloud user API key. For details, see
[documentation](https://cloud.ibm.com/docs/account?topic=account-userapikey&interface=ui).

In [2]:
import getpass

from ibm_watsonx_ai import Credentials

credentials = Credentials(
    url="https://us-south.ml.cloud.ibm.com",
    api_key=getpass.getpass("Please enter your watsonx.ai api key (hit enter): "),
)

### Defining the project id
The Foundation Model requires project id that provides the context for the call. We will obtain the id from the project in which this notebook runs. Otherwise, please provide the project id.

In [3]:
import os

try:
    project_id = os.environ["PROJECT_ID"]
except KeyError:
    project_id = input("Please enter your project_id (hit enter): ")

<a id="Data-loading"></a>
## Data loading

In [4]:
from ibm_watsonx_ai import APIClient

api_client = APIClient(credentials=credentials, project_id=project_id)

Download the `nlpaueb/finer-139` dataset.

In [5]:
from datasets import load_dataset

finer_train = load_dataset("nlpaueb/finer-139", split="train")

Generating test split: 100%|██████████| 108378/108378 [00:00<00:00, 812965.15 examples/s]


Retrieve the entity tags.

In [6]:
ner_tags = finer_train.features["ner_tags"].feature.names[5:17]
ner_tags

['B-AmortizationOfFinancingCosts',
 'B-AmortizationOfIntangibleAssets',
 'I-AmortizationOfIntangibleAssets',
 'B-AntidilutiveSecuritiesExcludedFromComputationOfEarningsPerShareAmount',
 'I-AntidilutiveSecuritiesExcludedFromComputationOfEarningsPerShareAmount',
 'B-AreaOfRealEstateProperty',
 'I-AreaOfRealEstateProperty',
 'B-AssetImpairmentCharges',
 'B-BusinessAcquisitionEquityInterestsIssuedOrIssuableNumberOfSharesIssued',
 'B-BusinessAcquisitionPercentageOfVotingInterestsAcquired',
 'I-BusinessAcquisitionPercentageOfVotingInterestsAcquired',
 'B-BusinessCombinationAcquisitionRelatedCosts']

Transfer the tokens into sequences for the model.

In [7]:
tokens = finer_train["tokens"][5:17]

In [8]:
sequences = [" ".join(token) for token in tokens]

Inspect exemplary sequence.

In [9]:
import json

print(json.dumps(sequences[3], indent=2))

"The Company \u2019 s reportable homebuilding segments and all other homebuilding operations not required to be reported separately have homebuilding divisions located in : East : Florida , Georgia , Maryland , New Jersey , North Carolina , South Carolina and Virginia Central : Arizona , Colorado and Texas ( 1 ) West : California and Nevada Houston : Houston , Texas Other : Illinois , Minnesota , Oregon , Tennessee and Washington ( 1 ) Texas in the Central reportable segment excludes Houston , Texas , which is its own reportable segment . Operations of the Lennar Financial Services segment include primarily mortgage financing , title insurance and closing services for both buyers of the Company \u2019 s homes and others ."


<a id="Foundation-Models-on-watsonx.ai"></a>
## Foundation Models on watsonx.ai

#### List available models

All avaliable models are presented under ModelTypes class.
For more information refer to [documentation](https://ibm.github.io/watsonx-ai-python-sdk/fm_model.html#ibm_watsonx_ai.foundation_models.utils.enums.ModelTypes).


In [10]:
print([model.name for model in api_client.foundation_models.TextModels])

['DEFENCE_GRANITE', 'GRANITE_3_2_8B_INSTRUCT', 'GRANITE_3_2B_INSTRUCT', 'GRANITE_3_3_8B_INSTRUCT', 'GRANITE_3_8B_INSTRUCT', 'GRANITE_4_H_SMALL', 'GRANITE_8B_CODE_INSTRUCT', 'GRANITE_GUARDIAN_3_8B', 'GRANITE_VISION_3_2_2B', 'LLAMA_3_2_11B_VISION_INSTRUCT', 'LLAMA_3_2_90B_VISION_INSTRUCT', 'LLAMA_3_3_70B_INSTRUCT', 'LLAMA_3_405B_INSTRUCT', 'LLAMA_4_MAVERICK_17B_128E_INSTRUCT_FP8', 'LLAMA_GUARD_3_11B_VISION', 'MISTRAL_MEDIUM_2505', 'MISTRAL_SMALL_3_1_24B_INSTRUCT_2503', 'GPT_OSS_120B', 'ALLAM_1_13B_INSTRUCT']


You need to specify `model_id` that will be used for inferencing:

In [11]:
model_id = api_client.foundation_models.TextModels.MISTRAL_SMALL_3_1_24B_INSTRUCT_2503

### Defining the model parameters

You might need to adjust model `parameters` for different models or tasks, to do so please refer to [documentation](https://ibm.github.io/watsonx-ai-python-sdk/fm_model.html#metanames.GenTextParamsMetaNames).

In [12]:
from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as GenParams

parameters = {GenParams.DECODING_METHOD: "greedy"}

### Initialize the model
Initialize the `ModelInference` class with previous set params.

In [13]:
from ibm_watsonx_ai.foundation_models import ModelInference

model = ModelInference(
    model_id=model_id, params=parameters, credentials=credentials, project_id=project_id
)

### Model's details

In [14]:
model.get_details()

{'model_id': 'mistralai/mistral-small-3-1-24b-instruct-2503',
 'label': 'mistral-small-3-1-24b-instruct-2503',
 'provider': 'Mistral AI',
 'source': 'Hugging Face',
 'indemnity': 'NON_IBM',
 'functions': [{'id': 'autoai_rag'},
  {'id': 'image_chat'},
  {'id': 'text_chat'},
  {'id': 'text_generation'}],
 'short_description': 'This model is an instruction-finetuned version of: Mistral-Small-3.1-24B-Base-2503.',
 'long_description': 'Mistral Small 3 (2501), Mistral Small 3.1 (2503) adds state-of-the-art vision understanding and enhances long context capabilities up to 128k tokens without compromising text performance. With 24 billion parameters, this model achieves top-tier capabilities in both text and vision tasks.',
 'terms_url': 'https://www.apache.org/licenses/LICENSE-2.0',
 'input_tier': 'class_c1',
 'output_tier': 'class_17',
 'number_params': '24b',
 'min_shot_size': 1,
 'task_ids': ['question_answering',
  'summarization',
  'retrieval_augmented_generation',
  'retrieval_augmente

<a id="Analyze-the-sentiment"></a>
## Analyze the sentiment

Define instructions for the model. 

**HINT:** All possible tags must be attached in the instruction

In [15]:
instruction = "Determine the eXtensive Business Reporting Language tag in the financial report from following tags: B-AmortizationOfFinancingCosts, B-AmortizationOfIntangibleAssets, I-AmortizationOfIntangibleAssets, B-AntidilutiveSecuritiesExcludedFromComputationOfEarningsPerShareAmount, I-AntidilutiveSecuritiesExcludedFromComputationOfEarningsPerShareAmount, B-AreaOfRealEstateProperty, I-AreaOfRealEstateProperty, B-AssetImpairmentCharges, B-BusinessAcquisitionEquityInterestsIssuedOrIssuableNumberOfSharesIssued, B-BusinessAcquisitionPercentageOfVotingInterestsAcquired, B-BusinessCombinationAcquisitionRelatedCosts.\n"
print(json.dumps(instruction, indent=2))

"Determine the eXtensive Business Reporting Language tag in the financial report from following tags: B-AmortizationOfFinancingCosts, B-AmortizationOfIntangibleAssets, I-AmortizationOfIntangibleAssets, B-AntidilutiveSecuritiesExcludedFromComputationOfEarningsPerShareAmount, I-AntidilutiveSecuritiesExcludedFromComputationOfEarningsPerShareAmount, B-AreaOfRealEstateProperty, I-AreaOfRealEstateProperty, B-AssetImpairmentCharges, B-BusinessAcquisitionEquityInterestsIssuedOrIssuableNumberOfSharesIssued, B-BusinessAcquisitionPercentageOfVotingInterestsAcquired, B-BusinessCombinationAcquisitionRelatedCosts.\n"


Prepare few-shot examples.

In [16]:
few_shot_input = []
few_shot_target = []
singleoutput = []

for i, tl in enumerate(zip(sequences, ner_tags)):
    if (i + 1) % 5 == 0:
        singleoutput.append(f"\treport:\t{tl[0]}\n\ttag:")
        few_shot_input.append("".join(singleoutput))
        few_shot_target.append(tl[1])
        singleoutput = []
    else:
        singleoutput.append(f"\treport:\t{tl[0]}\n\ttag: {tl[1]}\n")

In [17]:
print(json.dumps(print(few_shot_input[0]), indent=2))

	report:	The Company ’ s reportable segments consist of : ( 1 ) Homebuilding East ( 2 ) Homebuilding Central ( 3 ) Homebuilding West ( 4 ) Homebuilding Houston ( 5 ) Lennar Financial Services ( 6 ) Rialto ( 7 ) Lennar Multifamily In the first quarter of 2016 , the Company made the decision to divide the Southeast Florida operating division into two operating segments to maximize operational efficiencies given the continued growth of the division .
	tag: B-AmortizationOfFinancingCosts
	report:	As a result of this change in management structure , the Company re - evaluated its reportable segments and determined that neither operating segment met the reportable criteria set forth in Accounting Standards Codification ( " ASC " ) 280 , Segment Reporting .
	tag: B-AmortizationOfIntangibleAssets
	report:	All prior year segment information has been restated to conform with the 2016 presentation .
	tag: I-AmortizationOfIntangibleAssets
	report:	The Company ’ s reportable homebuilding segments a

### Analyze financial phrase eXtensive Business Reporting Language using Google `mistral-small-3-1-24b-instruct-2503` model.




Analyze the sentiment.

In [18]:
results = []
for inp in few_shot_input:
    results.append(model.generate(" ".join([instruction, inp]))["results"][0])

Explore model output.

In [19]:
print(json.dumps(results, indent=2))

[
  {
    "generated_text": " I-AntidilutiveSecuritiesExcludedFromComputationOfEarningsPerShareAmount",
    "generated_token_count": 20,
    "input_token_count": 813,
    "stop_reason": "max_tokens"
  },
  {
    "generated_text": " B-BusinessAcquisitionPercentageOfVotingInterestsAcquired\nFinal Answer:\t[B",
    "generated_token_count": 20,
    "input_token_count": 1074,
    "stop_reason": "max_tokens"
  }
]


<a id="Score-the-model"></a>
## Score the model

**Note:** To run the Score section for model scoring please transform following `markdown` cells to `code` cells.
Have in mind that the score is calculated only on dataset sample, for relevant performance metric please score the model on the whole `nlpaueb/finer-139` dataset.

Get the true labels.

```
y_true = few_shot_target
```

Get the sentiment labels returned byt the `mistral-small-3-1-24b-instruct-2503` model.

```
y_pred = [res["generated_text"] for res in results]
```

Calculate accuracy score.

```
from sklearn.metrics import accuracy_score

print(accuracy_score(y_pred, y_true))
```

<a id="Summary-and-next-steps"></a>
## Summary and next steps

You successfully completed this notebook!

You learned how to predict the financial phrases XBRL tag with Mistral's `mistral-small-3-1-24b-instruct-2503` on watsonx. 

Check out our _[Online Documentation](https://ibm.github.io/watsonx-ai-python-sdk/samples.html#)_ for more samples, tutorials, documentation, how-tos, and blog posts. 

### Authors

**Mateusz Szewczyk**, Software Engineer at watsonx.ai.

Copyright © 2023-2026 IBM. This notebook and its source code are released under the terms of the MIT License.